In [1]:
import os
import groq
groq.api_key = os.getenv("GROQ_API_KEY")

In [3]:
client = groq.Client()
model = "llama-3.3-70b-versatile"

def call_llm(input_text):
    text = '\n'.join(input_text.strip()) if isinstance(input_text, str) else input_text
    prompt = f"Please elaborate on the following content:\n{text}"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role":"system","content": "You are an expert in Natural Language Processing exercise professional. In your responses, do NOT cite the context provided after the sentence **augmented input context**"},
                {"role":"assistant", "content": "You must explain and answer in detail to the user query, paying attention to the provided context."},
                {"role":"user", "content": prompt}
            ],
            temperature=0.1
        )
    except Exception as e:
        return str(e)
    
    return response.choices[0].message.content.strip()

In [4]:
import requests
from bs4 import BeautifulSoup
import re

urls = [
    "https://en.wikipedia.org/wiki/Space_exploration",
    "https://en.wikipedia.org/wiki/Apollo_program",
    "https://en.wikipedia.org/wiki/Hubble_Space_Telescope",
    "https://en.wikipedia.org/wiki/Mars_rover",  # Corrected link
    "https://en.wikipedia.org/wiki/International_Space_Station",
    "https://en.wikipedia.org/wiki/SpaceX",
    "https://en.wikipedia.org/wiki/Juno_(spacecraft)",
    "https://en.wikipedia.org/wiki/Voyager_program",
    "https://en.wikipedia.org/wiki/Galileo_(spacecraft)",
    "https://en.wikipedia.org/wiki/Kepler_Space_Telescope",
    "https://en.wikipedia.org/wiki/James_Webb_Space_Telescope",
    "https://en.wikipedia.org/wiki/Space_Shuttle",
    "https://en.wikipedia.org/wiki/Artemis_program",
    "https://en.wikipedia.org/wiki/Skylab",
    "https://en.wikipedia.org/wiki/NASA",
    "https://en.wikipedia.org/wiki/European_Space_Agency",
    "https://en.wikipedia.org/wiki/Ariane_(rocket_family)",
    "https://en.wikipedia.org/wiki/Spitzer_Space_Telescope",
    "https://en.wikipedia.org/wiki/New_Horizons",
    "https://en.wikipedia.org/wiki/Cassini%E2%80%93Huygens",
    "https://en.wikipedia.org/wiki/Curiosity_(rover)",
    "https://en.wikipedia.org/wiki/Perseverance_(rover)",
    "https://en.wikipedia.org/wiki/InSight",
    "https://en.wikipedia.org/wiki/OSIRIS-REx",
    "https://en.wikipedia.org/wiki/Parker_Solar_Probe",
    "https://en.wikipedia.org/wiki/BepiColombo",
    "https://en.wikipedia.org/wiki/Juice_(spacecraft)",
    "https://en.wikipedia.org/wiki/Solar_Orbiter",
    "https://en.wikipedia.org/wiki/CHEOPS_(satellite)",
    "https://en.wikipedia.org/wiki/Gaia_(spacecraft)"
]

Preparing the data

In [5]:
def clean_text(text):
    content = re.sub(r'\d+', '', text)  # Remove numbers
    return content

def fetch_and_clean(url):
    # Use a more realistic, modern browser User-Agent
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the main content of the article, ignoring side boxes and headers
    content = soup.find('div', {'class': 'mw-parser-output'})
    
    # If content is not found (perhaps page structure is different or page failed to load), return empty string
    if content is None:
        print(f"Warning: Could not extract content from {url}")
        return ""

    # remove the bibliography section
    for section_title in ['References', 'External links', 'Further reading', 'Bibliography', 'See also']:
        section = content.find('span', id=section_title)
        if section:
            for sib in section.parent.find_next_siblings():
                sib.decompose()
            section.parent.decompose()

    text = content.get_text(separator=' ', strip=True)
    text = clean_text(text)
    return text

with open('llm.txt', 'w', encoding='utf-8') as f:
    for url in urls:
        clean_article_text = fetch_and_clean(url)
        if clean_article_text:
            f.write(clean_article_text + '\n')


print("Finished writing to llm.txt")

Finished writing to llm.txt


In [6]:

from pathlib import Path
import os

output_dir = Path(os.getcwd()) / 'data'
output_dir.mkdir(exist_ok=True)

In [7]:

file_path = output_dir / 'llm.txt'
#save content in 'llm.txt' for modularity
with open(file_path, 'w', encoding='utf-8') as f:
    for url in urls:
        f.write(fetch_and_clean(url) + '\n')

print(f'Content successfully written in {file_path}')

Content successfully written in c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch2\data\llm.txt


In [8]:
# Open the file and read the first 20 lines
with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()
    # Print the first 20 lines
    for line in lines[:20]:
        print(line.strip())

Investigation of space, planets, and moons For broader coverage of this topic, see Exploration . Buzz Aldrin taking a core sample of the Moon during the Apollo  mission Self-portrait of Curiosity rover on Mars 's surface Part of a series on Spaceflight History History of spaceflight Space Race Timeline of spaceflight Space probes Lunar missions Mars missions Applications Communications Earth observation Exploration Espionage Military Navigation Colonization Habitation Exploration Telescopes Tourism Spacecraft Robotic spacecraft Satellite Space probe Cargo spacecraft Crewed spacecraft Apollo Lunar Module Space capsules Space Shuttle Space stations Spaceplanes Vostok Space launch Spaceport Launch pad Expendable and reusable launch vehicles Escape velocity Non-rocket spacelaunch Spaceflight types Sub-orbital Orbital Interplanetary Interstellar Intergalactic List of space organizations Space agencies Space forces Companies Spaceflight portal v t e Space exploration is the physical investig

Enbedding and Storage

In [9]:
source_text = file_path

with open(source_text, 'r', encoding='utf-8') as f:
    text = f.read()

CHUNK_SIZE = 1000
chunked_text = [text[i:i+CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]

In [10]:

import deeplake
from deeplake.core.vectorstore.deeplake_vectorstore import VectorStore
import deeplake.util
import numpy as np


vector_path = str(output_dir / "dory_vectorstore")
print(f"Vector store path: {vector_path}")


vector_store = VectorStore(path=vector_path, overwrite=True)
print("Vector Store created successfully")

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.5.9) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


Vector store path: c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch2\data\dory_vectorstore
Vector Store created successfully


In [11]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

In [12]:
def embedding_function(texts, model=model):
    if isinstance(texts, str):
        texts = [texts]
    texts = [t.replace('\n', ' ') for t in texts]
    return [model.encode(t) for t in texts]

In [13]:
vector_store.add(text=chunked_text, #the text to be embedded
                 embedding_function=embedding_function,
                 embedding_data=chunked_text, #embedding_function embeds embedding_data to create embeddings
                 metadata=[{"source": str(source_text)}] * len(chunked_text)) #for each embedding the metadata is 'llm.txt'

Creating 1598 embeddings in 4 batches of size 500:: 100%|██████████| 4/4 [00:18<00:00,  4.70s/it]

Dataset(path='c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch2\data\dory_vectorstore', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype       shape      dtype  compression
  -------    -------     -------    -------  ------- 
   text       text      (1598, 1)     str     None   
 metadata     json      (1598, 1)     str     None   
 embedding  embedding  (1598, 384)  float32   None   
    id        text      (1598, 1)     str     None   


In [ ]:
print(vector_store.summary())
ds = deeplake.load(vector_path)
ds.visualize()

Dataset(path='c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch2\data\dory_vectorstore', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype       shape      dtype  compression
  -------    -------     -------    -------  ------- 
   text       text      (1598, 1)     str     None   
 metadata     json      (1598, 1)     str     None   
 embedding  embedding  (1598, 384)  float32   None   
    id        text      (1598, 1)     str     None   
None
c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch2\data\dory_vectorstore loaded successfully.



HINT: Please forward the port - 63339 to your local machine, if you are running on the cloud.


 * Serving Flask app 'dataset_visualizer'
 * Debug mode: off


In [15]:

ds_size = ds.size_approx()
ds_size_mb = ds_size / 1048576
print(f"Dataset size in MB: {ds_size_mb:5f} MB")

ds_size_gb = ds_size / 1073741824
print(f"Dataset size in GB: {ds_size_gb:5f} GB")

Dataset size in MB: 55.313110 MB
Dataset size in GB: 0.054017 GB


Augmented input for generation

In [20]:

def get_user_prompt():
    return input("Enter your search query")



user_prompt = get_user_prompt()

In [21]:
search_results = vector_store.search(embedding_data=user_prompt, embedding_function=embedding_function)

In [22]:
def wrap_text(text, width=80):
    lines = []
    while len(text) > width:
        split_index = text.rfind(' ', 0, width)
        if split_index == -1:
            split_index = width
        lines.append(text[:split_index])
        text = text[split_index:].strip()
    lines.append(text)
    return '\n'.join(lines)

In [23]:
print(user_prompt)
top_score = search_results['score'][0]
top_text = search_results['text'][0].strip()
top_metadata = search_results['metadata'][0]['source']

print("Top Search Result:")
print(f'Score: {top_score}')
print(f'Source: {top_metadata}')
print('Text:')
print(wrap_text(top_text, width=30))


What is the moon?
Top Search Result:
Score: 0.45656129717826843
Source: c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch2\data\llm.txt
Text:
a space-based perspective.
These satellites have
significantly contributed to
the understanding of a
variety of Earth-based
phenomena. For instance, the
hole in the ozone layer was
found by an artificial
satellite that was exploring
Earth's atmosphere, and
satellites have allowed for
the discovery of
archeological sites or
geological formations that
were difficult or impossible
to otherwise identify. Moon [
edit ] Main article:
Exploration of the Moon
Apollo  LEM Orion, the Lunar
Roving Vehicle and astronaut
John Young () The Moon was
the first celestial body to
be the object of space
exploration. It holds the
distinctions of being the
first remote celestial object
to be flown by, orbited, and
landed upon by spacecraft,
and the only remote celestial
object ever to be visited by
humans. In , the